# Notebook 4: Feature Engineering
## Car Price Prediction Project

**Objective:** Create new meaningful features from existing data to improve model performance.

### Feature Engineering Plan:
1. Brand tier (from brand name)
2. Ratios & proportions (physics-inspired)
3. Binning continuous variables
4. Interaction features
5. Polynomial / non-linear transforms
6. Aggregation-based features
7. Boolean / flag features
8. Drop redundant features

---

## 4.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/drive/MyDrive/car_price_cleaned.csv')
print(f"Loaded: {df.shape}")
df.head()

## 4.2 Brand Tier (Most Critical Feature)

Group brands into pricing tiers based on average price — this is the single most powerful categorical feature.

In [ ]:
# Check average price by brand
brand_avg_price = df.groupby('brand')['price'].mean().sort_values(ascending=False)
print("Average price by brand:")
print(brand_avg_price.to_string())

In [ ]:
# Define brand tiers
luxury_brands = ['bmw', 'porsche', 'jaguar', 'buick', 'audi']
mid_brands = ['volvo', 'saab', 'peugeot', 'mazda', 'mercury', 'alfa-romeo']
economy_brands = ['toyota', 'nissan', 'dodge', 'plymouth', 'chevrolet',
                   'subaru', 'volkswagen', 'honda', 'mitsubishi',
                   'isuzu', 'renault']

def get_brand_tier(brand):
    if brand in luxury_brands:
        return 'luxury'
    elif brand in mid_brands:
        return 'mid-range'
    else:
        return 'economy'

df['brand_tier'] = df['brand'].apply(get_brand_tier)

print("Brand tier distribution:")
print(df['brand_tier'].value_counts())
print(f"\nAverage price by tier:")
print(df.groupby('brand_tier')['price'].mean().sort_values(ascending=False))

In [ ]:
# Visualize brand tier vs price
plt.figure(figsize=(8, 5))
df.boxplot(column='price', by='brand_tier')
plt.title('Price Distribution by Brand Tier')
plt.suptitle('')
plt.ylabel('Price ($)')
plt.tight_layout()
plt.show()

## 4.3 Ratios & Proportions (Physics-Inspired)

In [ ]:
# Power-to-weight ratio — separates sports cars from heavy sedans
df['power_to_weight'] = df['horsepower'] / df['curbweight']

# Engine-to-weight ratio — how oversized the engine is for the car
df['engine_to_weight'] = df['enginesize'] / df['curbweight']

# Bore-stroke ratio — engine design: oversquare (>1) = power, undersquare (<1) = torque
df['bore_stroke_ratio'] = df['boreratio'] / df['stroke']

# Fuel efficiency ratio — how well efficiency holds on highway vs city
df['fuel_efficiency_ratio'] = df['highwaympg'] / df['citympg']

# Length-width ratio — body shape proxy (lower = wider/sportier)
df['length_width_ratio'] = df['carlength'] / df['carwidth']

# Displacement per cylinder
df['displacement_per_cyl'] = df['enginesize'] / df['cylindernumber']

print("New ratio features created:")
ratio_cols = ['power_to_weight', 'engine_to_weight', 'bore_stroke_ratio',
              'fuel_efficiency_ratio', 'length_width_ratio', 'displacement_per_cyl']
df[ratio_cols].describe().T

In [ ]:
# Correlation of new ratio features with price
print("Correlation of new ratio features with price:")
for col in ratio_cols:
    r = df[col].corr(df['price'])
    print(f"  {col}: {r:.3f}")

## 4.4 Binning Continuous Variables

In [ ]:
# Horsepower bins
df['hp_bin'] = pd.cut(df['horsepower'],
                       bins=[0, 80, 150, 999],
                       labels=['low', 'medium', 'high'])

# Engine size bins
df['enginesize_bin'] = pd.cut(df['enginesize'],
                               bins=[0, 100, 180, 999],
                               labels=['small', 'medium', 'large'])

# Curbweight bins
df['weight_bin'] = pd.cut(df['curbweight'],
                           bins=[0, 2200, 2800, 9999],
                           labels=['light', 'medium', 'heavy'])

print("Horsepower bins:")
print(df['hp_bin'].value_counts())
print(f"\nEngine size bins:")
print(df['enginesize_bin'].value_counts())
print(f"\nWeight bins:")
print(df['weight_bin'].value_counts())

In [ ]:
# Average price by bins
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['hp_bin', 'enginesize_bin', 'weight_bin']):
    df.groupby(col)['price'].mean().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Average Price by {col}')
    ax.set_ylabel('Price ($)')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 4.5 Interaction Features

In [ ]:
# Is turbo AND rear-wheel drive → sports/performance car flag
df['is_turbo_rwd'] = ((df['aspiration'] == 'turbo') & (df['drivewheel'] == 'rwd')).astype(int)

# Is diesel sedan → specific segment
df['is_diesel_sedan'] = ((df['fueltype'] == 'diesel') & (df['carbody'] == 'sedan')).astype(int)

# Is convertible RWD → luxury sports flag
df['is_convertible_rwd'] = ((df['carbody'] == 'convertible') & (df['drivewheel'] == 'rwd')).astype(int)

# FWD economy flag → most common cheap car profile
df['fwd_economy_flag'] = ((df['drivewheel'] == 'fwd') &
                           (df['cylindernumber'] == 4) &
                           (df['fueltype'] == 'gas')).astype(int)

print("Interaction feature counts:")
for col in ['is_turbo_rwd', 'is_diesel_sedan', 'is_convertible_rwd', 'fwd_economy_flag']:
    print(f"  {col}: {df[col].sum()} cars flagged")

In [ ]:
# Average price for interaction flags
for col in ['is_turbo_rwd', 'is_diesel_sedan', 'is_convertible_rwd', 'fwd_economy_flag']:
    avg = df.groupby(col)['price'].mean()
    print(f"\n{col}:")
    print(f"  Flag=0 avg price: ${avg.get(0, 0):,.0f}")
    print(f"  Flag=1 avg price: ${avg.get(1, 0):,.0f}")

## 4.6 Polynomial / Non-Linear Transforms

Address skewness and create non-linear features.

In [ ]:
# Log transforms for skewed features
df['log_price'] = np.log1p(df['price'])
df['log_enginesize'] = np.log1p(df['enginesize'])
df['log_horsepower'] = np.log1p(df['horsepower'])

# Enginesize squared — captures quadratic price acceleration for bigger engines
df['enginesize_sq'] = df['enginesize'] ** 2

# 1/citympg = fuel consumption (gallons per mile) — more linear with price
df['fuel_consumption'] = 1 / df['citympg']

print("Non-linear features created.")
print(f"\nCorrelation with price:")
for col in ['log_enginesize', 'log_horsepower', 'enginesize_sq', 'fuel_consumption']:
    print(f"  {col}: {df[col].corr(df['price']):.3f}")

In [ ]:
# Compare original vs transformed correlations
print("Original vs Transformed correlations with price:")
print(f"  enginesize:     {df['enginesize'].corr(df['price']):.3f}  →  log: {df['log_enginesize'].corr(df['price']):.3f}  |  squared: {df['enginesize_sq'].corr(df['price']):.3f}")
print(f"  horsepower:     {df['horsepower'].corr(df['price']):.3f}  →  log: {df['log_horsepower'].corr(df['price']):.3f}")
print(f"  citympg:        {df['citympg'].corr(df['price']):.3f}  →  1/citympg: {df['fuel_consumption'].corr(df['price']):.3f}")

## 4.7 Aggregation-Based Features (Target Encoding Style)

**Warning:** These must be calculated carefully to avoid data leakage. We compute them on the full training set here for demonstration. In production, use cross-validated encoding.

In [ ]:
# Brand average price — encodes brand prestige as a number
brand_price_map = df.groupby('brand')['price'].mean()
df['brand_avg_price'] = df['brand'].map(brand_price_map)

# Brand count — popular vs niche brands
brand_count_map = df.groupby('brand')['price'].count()
df['brand_count'] = df['brand'].map(brand_count_map)

# Body type average HP
body_hp_map = df.groupby('carbody')['horsepower'].mean()
df['bodytype_avg_hp'] = df['carbody'].map(body_hp_map)

# Drivewheel average weight
dw_weight_map = df.groupby('drivewheel')['curbweight'].mean()
df['drivewheel_avg_weight'] = df['drivewheel'].map(dw_weight_map)

print("Aggregation features created:")
agg_cols = ['brand_avg_price', 'brand_count', 'bodytype_avg_hp', 'drivewheel_avg_weight']
for col in agg_cols:
    print(f"  {col}: corr with price = {df[col].corr(df['price']):.3f}")

## 4.8 Boolean / Flag Features

In [ ]:
# Is turbo aspiration
df['is_turbo'] = (df['aspiration'] == 'turbo').astype(int)

# Is luxury brand
df['is_luxury_brand'] = (df['brand_tier'] == 'luxury').astype(int)

# Has many cylinders (8 or 12)
df['has_many_cylinders'] = (df['cylindernumber'] >= 8).astype(int)

# Is gas fuel
df['is_gas'] = (df['fueltype'] == 'gas').astype(int)

# Is rear-wheel drive
df['is_rwd'] = (df['drivewheel'] == 'rwd').astype(int)

print("Boolean flag features:")
flag_cols = ['is_turbo', 'is_luxury_brand', 'has_many_cylinders', 'is_gas', 'is_rwd']
for col in flag_cols:
    print(f"  {col}: {df[col].sum()} flagged, corr with price = {df[col].corr(df['price']):.3f}")

## 4.9 Drop Redundant Features

In [ ]:
# Drop one of highly correlated pairs
# citympg & highwaympg: 0.97 correlation → keep citympg, drop highwaympg
# carlength & wheelbase: 0.88 correlation → keep carlength, drop wheelbase

cols_to_drop = ['highwaympg', 'wheelbase']

# Also consider dropping low-correlation features
low_corr_features = ['stroke', 'compressionratio', 'peakrpm']
for col in low_corr_features:
    r = abs(df[col].corr(df['price']))
    print(f"  {col}: |corr with price| = {r:.3f}")

cols_to_drop.extend(low_corr_features)

print(f"\nDropping: {cols_to_drop}")
df.drop(columns=cols_to_drop, inplace=True)
print(f"Shape after dropping: {df.shape}")

## 4.10 Final Feature Overview

In [ ]:
print(f"Final dataset shape: {df.shape}")
print(f"\nAll columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    if dtype in ['float64', 'int64']:
        r = df[col].corr(df['price'])
        print(f"  {i:2d}. {col:<30s} {str(dtype):<10s} corr={r:.3f}")
    else:
        print(f"  {i:2d}. {col:<30s} {str(dtype):<10s}")

In [ ]:
# Top 15 features correlated with price
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'price' in num_cols:
    num_cols.remove('price')
if 'log_price' in num_cols:
    num_cols.remove('log_price')

corr_with_price = df[num_cols].corrwith(df['price']).abs().sort_values(ascending=False)
print("Top 15 features by |correlation| with price:")
print(corr_with_price.head(15).to_string())

## 4.11 Save Engineered Dataset

In [ ]:
df.to_csv('/content/drive/MyDrive/car_price_engineered.csv', index=False)
print("Feature-engineered data saved to 'car_price_engineered.csv'")

---
## Feature Engineering Summary

| Category | Features Created |
|----------|------------------|
| **Brand Tier** | `brand_tier` (luxury/mid-range/economy) |
| **Ratios** | `power_to_weight`, `engine_to_weight`, `bore_stroke_ratio`, `fuel_efficiency_ratio`, `length_width_ratio`, `displacement_per_cyl` |
| **Bins** | `hp_bin`, `enginesize_bin`, `weight_bin` |
| **Interactions** | `is_turbo_rwd`, `is_diesel_sedan`, `is_convertible_rwd`, `fwd_economy_flag` |
| **Transforms** | `log_price`, `log_enginesize`, `log_horsepower`, `enginesize_sq`, `fuel_consumption` |
| **Aggregation** | `brand_avg_price`, `brand_count`, `bodytype_avg_hp`, `drivewheel_avg_weight` |
| **Flags** | `is_turbo`, `is_luxury_brand`, `has_many_cylinders`, `is_gas`, `is_rwd` |
| **Dropped** | `highwaympg`, `wheelbase`, `stroke`, `compressionratio`, `peakrpm` |

**Priority features:** `brand_tier`, `power_to_weight`, `log(price)`, `fuel_consumption`, `bore_stroke_ratio`

**Next step →** Notebook 05: Preprocessing